# Разработка и исследование трансформерной модели для автоматического решения учебных тестовых заданий по медицинским дисциплинам

## Научно-практическое исследование

**Предметная область:** медицинское образование  
**Задача:** автоматический выбор правильного ответа в русскоязычных медицинских тестовых заданиях  
**Метод:** fine-tuning Transformer-модели для Multiple Choice Question Answering  
**Базовая модель:** `cointegrated/rubert-tiny2`  
**Датасет:** `drkolesnikov/russian-nmo-medical-mcq`

Работа рассматривает применение методов NLP в образовательном процессе медицинского колледжа.
Разрабатываемая система предназначена только для **учебных тестов** на базе медицинского колледжа !!

## 1. Актуальность

Медицинское образование требует усвоения большого объёма специализированной терминологии,
причинно-следственных связей и предметных знаний. При этом электронные образовательные системы
содержат всё больше тестов и контрольных заданий.

Transformer-модели способны учитывать контекст вопроса и вариантов ответа. Поэтому практический
интерес представляет задача автоматического решения учебных медицинских тестов и последующего
анализа тем, вызывающих затруднения.

**Исследовательский вопрос:** сможет ли компактная русскоязычная Transformer-модель решать медицинские MCQ лучше простых baseline-методов?

## 2. Цель, задачи и гипотеза

### Цель
Разработать и исследовать Transformer-модель для автоматического выбора правильного ответа
в русскоязычных тестовых заданиях по медицинским дисциплинам.

### Задачи
1. Загрузить открытый медицинский MCQ-датасет.
2. Выполнить фильтрацию и EDA.
3. Построить случайный и классический NLP baseline.
4. Подготовить задачу Multiple Choice для RuBERT.
5. Выполнить fine-tuning.
6. Оценить Accuracy, Precision, Recall и F1 на независимом test split.
7. Исследовать качество по медицинским темам.
8. Выполнить анализ ошибок модели.

### Гипотеза
> Специализированное дообучение русскоязычной Transformer-модели на медицинских тестах
> позволит получить качество выше случайного и классического текстового baseline.

## 3. Дизайн эксперимента

Используем исходные независимые `train / validation / test` split датасета.
В работу включаем только задания с **одним правильным ответом и четырьмя вариантами**.

`validation` используется для выбора лучшего checkpoint, а `test` — только для финальной оценки.
Чтобы эксперимент был выполним в Google Colab, берём фиксированную воспроизводимую подвыборку.
Размеры можно увеличить в конфигурационной ячейке.

## 4. Установка библиотек

In [ ]:
%pip install -q     "transformers>=4.57,<5"     "datasets>=3,<5"     "accelerate>=1,<2"     "scikit-learn>=1.4"     "pandas>=2"     "matplotlib>=3.8"

## 5. Импорты и воспроизводимость

In [ ]:
# Импорты

import inspect
import random
import shutil
import warnings
from collections import Counter
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from datasets import DatasetDict, load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline

from transformers import (
    AutoModelForMultipleChoice,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 6. Конфигурация исследования

Все основные параметры вынесены в одну ячейку. Это упрощает воспроизводимость и масштабирование эксперимента.

In [ ]:
#===
# Конфигурация

DATASET_NAME = "drkolesnikov/russian-nmo-medical-mcq"
DATASET_CONFIG = "mcq"
MODEL_NAME = "cointegrated/rubert-tiny2"

TRAIN_SIZE = 12000
VALID_SIZE = 2000
TEST_SIZE = 2000
EDA_SIZE = 30000

MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 3e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "/content/medical_mcq_rubert_tiny2"

print("Train:", TRAIN_SIZE)
print("Validation:", VALID_SIZE)
print("Test:", TEST_SIZE)

## 7. Датасет

Используется `drkolesnikov/russian-nmo-medical-mcq`, конфигурация `mcq`.
Структурированный вариант содержит поля `topic`, `question`, `choices`,
`correct_choice_indices`, `choice_count`, `is_multi_answer` и другие.

In [ ]:
# Загрузка датасета

raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(raw_dataset)

split_sizes = pd.DataFrame({
    "split": list(raw_dataset.keys()),
    "objects": [len(raw_dataset[s]) for s in raw_dataset.keys()],
})
display(split_sizes)

## 8. Проверка реальной структуры объекта

In [ ]:
# Пример объекта

example = raw_dataset["train"][0]
for key, value in example.items():
    print(f"{key}: {value}")

## 9. Фильтрация

Оставляем только single-choice вопросы с четырьмя вариантами ответа.
Индекс правильного ответа должен находиться в диапазоне `0..3`.

In [ ]:
# Фильтр

def keep_example(example):
    indices = example["correct_choice_indices"]
    return (
        not bool(example["is_multi_answer"])
        and int(example["choice_count"]) == 4
        and int(example["correct_choice_count"]) == 1
        and len(indices) == 1
        and int(indices[0]) in (0, 1, 2, 3)
    )

filtered_dataset = DatasetDict({
    split: raw_dataset[split].filter(keep_example)
    for split in raw_dataset.keys()
})

filtered_sizes = pd.DataFrame({
    "split": list(filtered_dataset.keys()),
    "objects_after_filter": [len(filtered_dataset[s]) for s in filtered_dataset.keys()],
})
display(filtered_sizes)

## 10. Формирование воспроизводимой экспериментальной выборки

In [ ]:
# Подвыборка

def take_sample(dataset, size, seed):
    size = min(size, len(dataset))
    return dataset.shuffle(seed=seed).select(range(size))

train_ds = take_sample(filtered_dataset["train"], TRAIN_SIZE, SEED)
valid_ds = take_sample(filtered_dataset["validation"], VALID_SIZE, SEED + 1)
test_ds = take_sample(filtered_dataset["test"], TEST_SIZE, SEED + 2)

print("Train:", len(train_ds))
print("Validation:", len(valid_ds))
print("Test:", len(test_ds))

# Часть 1. Разведочный анализ данных — EDA

In [ ]:
# EDA-выборка

eda_ds = take_sample(filtered_dataset["train"], EDA_SIZE, SEED + 10)

eda_df = pd.DataFrame({
    "topic": eda_ds["topic"],
    "question": eda_ds["question"],
    "choices": eda_ds["choices"],
    "label": [int(x[0]) for x in eda_ds["correct_choice_indices"]],
})

eda_df["question_chars"] = eda_df["question"].astype(str).str.len()
eda_df["mean_choice_chars"] = eda_df["choices"].apply(
    lambda choices: np.mean([len(str(choice)) for choice in choices])
)

display(eda_df.head())

## 11. Распределение позиции правильного ответа

In [ ]:
# Позиции правильного ответа

label_counts = eda_df["label"].value_counts().sort_index()
label_table = pd.DataFrame({
    "answer_position": label_counts.index,
    "count": label_counts.values,
    "share": label_counts.values / label_counts.values.sum(),
})
display(label_table)

plt.figure(figsize=(8, 5))
plt.bar(label_table["answer_position"].astype(str), label_table["count"])
plt.xlabel("Индекс правильного ответа")
plt.ylabel("Количество вопросов")
plt.title("Распределение позиции правильного ответа")
plt.grid(axis="y", alpha=0.3)
plt.show()

Если распределение близко к равномерному, нельзя получить высокую Accuracy,
просто всегда выбирая одну и ту же позицию ответа.

## 12. Длина вопросов

In [ ]:
# Длина вопросов

display(eda_df["question_chars"].describe().to_frame(name="question_chars"))

plt.figure(figsize=(10, 5))
plt.hist(eda_df["question_chars"], bins=50)
plt.xlabel("Количество символов")
plt.ylabel("Количество вопросов")
plt.title("Распределение длины медицинских вопросов")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 13. Длина вариантов ответа

In [ ]:
# Длина вариантов

plt.figure(figsize=(10, 5))
plt.hist(eda_df["mean_choice_chars"], bins=50)
plt.xlabel("Средняя длина варианта ответа")
plt.ylabel("Количество вопросов")
plt.title("Распределение длины вариантов ответа")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 14. Наиболее представленные медицинские темы

In [ ]:
# Рейтинг тем

TOP_N_TOPICS = 15
topic_counts = eda_df["topic"].value_counts().head(TOP_N_TOPICS).sort_values()

display(topic_counts.sort_values(ascending=False).to_frame(name="questions"))

plt.figure(figsize=(11, 7))
plt.barh(topic_counts.index, topic_counts.values)
plt.xlabel("Количество вопросов")
plt.ylabel("Тема")
plt.title("Наиболее представленные медицинские темы")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# Часть 2. Baseline-модели

Используем три точки сравнения:

1. случайный ответ;
2. всегда самая частая позиция;
3. классический `TF-IDF + линейный классификатор`.

Так Transformer сравнивается не только со случайным выбором, но и с разумным классическим NLP-методом.

In [ ]:
# Истинные ответы и метрики

y_train = np.array([int(x[0]) for x in train_ds["correct_choice_indices"]])
y_test = np.array([int(x[0]) for x in test_ds["correct_choice_indices"]])

def classification_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

## 15. Random baseline

In [ ]:
# Random baseline

rng = np.random.default_rng(SEED)
random_pred = rng.integers(0, 4, size=len(y_test))
random_metrics = classification_metrics(y_test, random_pred)
random_metrics

При четырёх равновероятных вариантах теоретическая Accuracy случайного ответа около `0.25`.

## 16. Majority-position baseline

In [ ]:
# Majority baseline

majority_label = Counter(y_train).most_common(1)[0][0]
majority_pred = np.full(len(y_test), majority_label)
majority_metrics = classification_metrics(y_test, majority_pred)

print("Самая частая позиция:", majority_label)
majority_metrics

## 17. TF-IDF baseline

Каждый вопрос разворачивается в четыре пары `вопрос [SEP] вариант`.
Линейная модель оценивает вероятность правильности каждой пары, после чего выбирается вариант
с максимальной вероятностью.

In [ ]:
# Подготовка пар для TF-IDF

def expand_mcq(dataset):
    texts = []
    labels = []
    for example in dataset:
        correct = int(example["correct_choice_indices"][0])
        question = str(example["question"])
        for index, choice in enumerate(example["choices"]):
            texts.append(question + " [SEP] " + str(choice))
            labels.append(int(index == correct))
    return texts, np.array(labels)

tfidf_train_texts, tfidf_train_y = expand_mcq(train_ds)
tfidf_test_texts, _ = expand_mcq(test_ds)

print("Train pairs:", len(tfidf_train_texts))
print("Test pairs:", len(tfidf_test_texts))

In [ ]:
# TF-IDF + SGD

tfidf_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
        ),
    ),
    (
        "classifier",
        SGDClassifier(
            loss="log_loss",
            class_weight="balanced",
            max_iter=30,
            tol=1e-3,
            random_state=SEED,
        ),
    ),
])

tfidf_model.fit(tfidf_train_texts, tfidf_train_y)

positive_probability = tfidf_model.predict_proba(tfidf_test_texts)[:, 1]
tfidf_scores = positive_probability.reshape(-1, 4)
tfidf_pred = np.argmax(tfidf_scores, axis=1)
tfidf_metrics = classification_metrics(y_test, tfidf_pred)

tfidf_metrics

# Часть 3. Transformer

## 18. Выбор модели

Используем `cointegrated/rubert-tiny2` — компактный русскоязычный BERT.
Для каждого вопроса модель получает четыре пары `(question, choice)` и возвращает четыре logits.

In [ ]:
# Токенизатор

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)
print("Vocabulary:", tokenizer.vocab_size)

## 19. Токенизация Multiple Choice

In [ ]:
# Препроцессинг

def preprocess_mcq(examples):
    first_sentences = []
    second_sentences = []

    for question, choices in zip(examples["question"], examples["choices"]):
        first_sentences.extend([question] * 4)
        second_sentences.extend(choices)

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    result = {
        key: [values[i:i + 4] for i in range(0, len(values), 4)]
        for key, values in tokenized.items()
    }
    result["labels"] = [int(x[0]) for x in examples["correct_choice_indices"]]
    return result


tokenized_train = train_ds.map(
    preprocess_mcq,
    batched=True,
    remove_columns=train_ds.column_names,
)
tokenized_valid = valid_ds.map(
    preprocess_mcq,
    batched=True,
    remove_columns=valid_ds.column_names,
)
tokenized_test = test_ds.map(
    preprocess_mcq,
    batched=True,
    remove_columns=test_ds.column_names,
)

print(tokenized_train)

## 20. Dynamic padding

Collator разворачивает четыре варианта, выполняет padding и собирает tensor обратно в форму
`batch × 4 choices × sequence_length`.

In [ ]:
# Data collator

@dataclass
class MultipleChoiceCollator:
    tokenizer: object
    max_length: int = MAX_LENGTH

    def __call__(self, features):
        labels = [feature["labels"] for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened = []
        for feature in features:
            item = {k: v for k, v in feature.items() if k != "labels"}
            for choice_index in range(num_choices):
                flattened.append({
                    key: value[choice_index]
                    for key, value in item.items()
                })

        batch = self.tokenizer.pad(
            flattened,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        batch = {
            key: value.view(batch_size, num_choices, -1)
            for key, value in batch.items()
        }
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch


data_collator = MultipleChoiceCollator(tokenizer=tokenizer)

## 21. Проверка batch

In [ ]:
# Проверка batch

sample_batch = data_collator([tokenized_train[i] for i in range(2)])
for key, value in sample_batch.items():
    print(key, tuple(value.shape))

Для `input_ids` ожидается форма `(2, 4, sequence_length)`: два вопроса по четыре варианта.

## 22. Модель

In [ ]:
# RuBERT для Multiple Choice

model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model.to(DEVICE)

print(type(model).__name__)
print("Параметров:", sum(p.numel() for p in model.parameters()))

К предобученному BERT автоматически добавляется новая multiple-choice голова.
Её веса инициализируются заново и обучаются вместе с моделью.

## 23. Метрики

In [ ]:
# Метрики Trainer

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return classification_metrics(labels, predictions)

## 24. Параметры fine-tuning

Лучший checkpoint выбирается по `validation Accuracy`. Test split не участвует в выборе модели.

In [ ]:
# TrainingArguments

training_kwargs = {
    "output_dir": OUTPUT_DIR,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": 0.1,
    "logging_steps": 50,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "accuracy",
    "greater_is_better": True,
    "save_total_limit": 2,
    "report_to": [],
    "fp16": torch.cuda.is_available(),
    "seed": SEED,
}

signature = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in signature.parameters:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

In [ ]:
# Trainer

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_valid,
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_signature = inspect.signature(Trainer.__init__)
if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

# Часть 4. Fine-tuning

## 25. Обучение

На каждом шаге модель получает вопрос и четыре варианта, вычисляет четыре logits и обновляет веса
по ошибке выбора правильной позиции.

In [ ]:
# Fine-tuning

train_result = trainer.train()

## 26. Validation после обучения

In [ ]:
# Validation

validation_metrics = trainer.evaluate()
validation_metrics

## 27. Графики обучения

In [ ]:
# История обучения

history_df = pd.DataFrame(trainer.state.log_history)
display(history_df.tail(10))

train_history = history_df[history_df["loss"].notna()][["step", "loss"]]

plt.figure(figsize=(10, 5))
plt.plot(train_history["step"], train_history["loss"])
plt.xlabel("Шаг обучения")
plt.ylabel("Loss")
plt.title("Динамика train loss")
plt.grid(alpha=0.3)
plt.show()

if "eval_loss" in history_df.columns:
    eval_history = history_df[history_df["eval_loss"].notna()]
    if len(eval_history) > 0:
        plt.figure(figsize=(8, 5))
        plt.plot(
            range(1, len(eval_history) + 1),
            eval_history["eval_loss"],
            marker="o",
        )
        plt.xlabel("Проверка")
        plt.ylabel("Validation loss")
        plt.title("Динамика validation loss")
        plt.grid(alpha=0.3)
        plt.show()

# Часть 5. Финальная оценка

## 28. Test

Test split до этого момента не использовался для выбора checkpoint.

In [ ]:
# Test prediction

test_output = trainer.predict(tokenized_test)
test_logits = (
    test_output.predictions[0]
    if isinstance(test_output.predictions, tuple)
    else test_output.predictions
)

transformer_pred = np.argmax(test_logits, axis=1)
transformer_metrics = classification_metrics(y_test, transformer_pred)
transformer_metrics

## 29. Сравнение методов

In [ ]:
# Итоговая таблица

results_table = pd.DataFrame([
    {"model": "Random baseline", **random_metrics},
    {"model": "Majority-position", **majority_metrics},
    {"model": "TF-IDF + SGD", **tfidf_metrics},
    {"model": "Fine-tuned RuBERT", **transformer_metrics},
])

display(results_table.sort_values("accuracy", ascending=False).reset_index(drop=True))

accuracy_plot = results_table.sort_values("accuracy")
plt.figure(figsize=(9, 5))
plt.barh(accuracy_plot["model"], accuracy_plot["accuracy"])
plt.xlabel("Accuracy")
plt.ylabel("Метод")
plt.title("Сравнение качества решения медицинских MCQ")
plt.xlim(0, 1)
plt.grid(axis="x", alpha=0.3)
plt.show()

## 30. Проверка гипотезы

In [ ]:
# Проверка гипотезы

best_baseline_accuracy = max(
    random_metrics["accuracy"],
    majority_metrics["accuracy"],
    tfidf_metrics["accuracy"],
)

delta = transformer_metrics["accuracy"] - best_baseline_accuracy

print("Лучший baseline:", round(best_baseline_accuracy, 4))
print("Transformer:", round(transformer_metrics["accuracy"], 4))
print("Прирост:", round(delta, 4))

if delta > 0:
    print("Результаты данного эксперимента поддерживают гипотезу.")
else:
    print("В данном эксперименте гипотеза не получила подтверждения.")

## 31. Confusion matrix

In [ ]:
# Матрица ошибок

ConfusionMatrixDisplay.from_predictions(y_test, transformer_pred)
plt.title("Confusion matrix: позиция правильного ответа")
plt.grid(False)
plt.show()

Матрица показывает, не сформировалось ли систематическое предпочтение отдельных позиций ответа.

# Часть 6. Анализ по медицинским темам

In [ ]:
# DataFrame предсказаний

probabilities = torch.softmax(torch.tensor(test_logits), dim=1).numpy()
confidence = np.max(probabilities, axis=1)

test_analysis = pd.DataFrame({
    "topic": test_ds["topic"],
    "question": test_ds["question"],
    "choices": test_ds["choices"],
    "true_label": y_test,
    "pred_label": transformer_pred,
    "confidence": confidence,
})

test_analysis["correct"] = (
    test_analysis["true_label"] == test_analysis["pred_label"]
)

display(test_analysis.head())

In [ ]:
# Accuracy по темам

topic_metrics = (
    test_analysis
    .groupby("topic")
    .agg(
        questions=("correct", "size"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
)

MIN_TOPIC_OBJECTS = 10
reliable_topics = (
    topic_metrics[topic_metrics["questions"] >= MIN_TOPIC_OBJECTS]
    .sort_values("questions", ascending=False)
)

display(reliable_topics.head(20))

## 32. Качество наиболее представленных тем

In [ ]:
# График по темам

topic_plot = reliable_topics.head(15).sort_values("accuracy")

if len(topic_plot) > 0:
    plt.figure(figsize=(11, 7))
    plt.barh(topic_plot["topic"], topic_plot["accuracy"])
    plt.xlabel("Accuracy")
    plt.ylabel("Медицинская тема")
    plt.title("Качество Transformer по медицинским темам")
    plt.xlim(0, 1)
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

Различия между темами нужно интерпретировать осторожно: темы имеют разный объём и сложность,
а Accuracy небольших групп обладает высокой статистической неопределённостью.

# Часть 7. Анализ ошибок

## 33. Наиболее уверенные ошибки

Уверенные ошибки особенно полезны: они помогают найти сложные формулировки, близкие медицинские
термины, недостаток предметных знаний или неоднозначные задания.

In [ ]:
# Уверенные ошибки

errors = (
    test_analysis[~test_analysis["correct"]]
    .sort_values("confidence", ascending=False)
    .reset_index(drop=True)
)

print("Ошибок:", len(errors))
display(errors.head(10)[[
    "topic",
    "question",
    "true_label",
    "pred_label",
    "confidence",
]])

In [ ]:
# Подробный разбор ошибок

N_ERRORS_TO_SHOW = 5

for index in range(min(N_ERRORS_TO_SHOW, len(errors))):
    row = errors.iloc[index]
    print()
    print("=" * 70)
    print("Тема:", row["topic"])
    print("Вопрос:", row["question"])
    print()

    for choice_index, choice in enumerate(row["choices"]):
        marker = ""
        if choice_index == row["true_label"]:
            marker += " <- правильный"
        if choice_index == row["pred_label"]:
            marker += " <- модель"
        print(f"{choice_index}. {choice}{marker}")

    print("Уверенность:", round(float(row["confidence"]), 4))

## 34. Уверенные правильные ответы

In [ ]:
# Правильные примеры

correct_examples = (
    test_analysis[test_analysis["correct"]]
    .sort_values("confidence", ascending=False)
    .head(5)
)

display(correct_examples[[
    "topic",
    "question",
    "true_label",
    "confidence",
]])

# Часть 8. Практическая значимость

Результат может рассматриваться как прототип образовательного компонента медицинского колледжа:

```text
студент
   ->
учебное тестирование
   ->
анализ ошибок по темам
   ->
выявление слабых разделов
   ->
дополнительные задания
```

Модель не является клинической системой. Для реального образовательного внедрения необходима отдельная педагогическая и экспертная валидация.

# Часть 9. Ограничения

1. Используется подвыборка большого датасета.
2. Исследуются только вопросы с четырьмя вариантами и одним правильным ответом.
3. Результат зависит от состава обучающих данных.
4. Компактная модель выбрана с учётом ресурсов Colab.
5. Accuracy теста не эквивалентна клинической компетентности.
6. В исходных учебных данных могут встречаться ошибки или неоднозначности.

# Часть 10. Выводы

В исследовании построен полный pipeline:

```text
датасет → фильтрация → EDA → baselines → RuBERT → fine-tuning → test → анализ тем → анализ ошибок
```

Главное сравнение:

```text
Random vs Majority vs TF-IDF vs Fine-tuned Transformer
```

Численные выводы должны формироваться только после реального выполнения эксперимента.

## 35. Автоматический итог исследования

In [ ]:
# Итоговый вывод

baseline_rows = results_table[results_table["model"] != "Fine-tuned RuBERT"]
best_baseline_row = baseline_rows.sort_values("accuracy", ascending=False).iloc[0]

print("ИТОГ ИССЛЕДОВАНИЯ")
print("-" * 60)
print("Лучший baseline:", best_baseline_row["model"])
print("Accuracy baseline:", round(float(best_baseline_row["accuracy"]), 4))
print("Accuracy Transformer:", round(float(transformer_metrics["accuracy"]), 4))
print("F1 Transformer:", round(float(transformer_metrics["f1_macro"]), 4))

final_delta = transformer_metrics["accuracy"] - float(best_baseline_row["accuracy"])
print("Прирост Accuracy:", round(float(final_delta), 4))

if final_delta > 0:
    print("Fine-tuning Transformer превзошёл лучший baseline.")
    print("Результаты данного эксперимента поддерживают гипотезу.")
else:
    print("Transformer не превзошёл лучший baseline.")
    print("На данной выборке гипотеза не получила подтверждения.")

# Часть 11. Сохранение результатов

In [ ]:
# Сохранение модели и таблиц

FINAL_MODEL_DIR = "/content/medical_mcq_final_model"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

results_table.to_csv(
    "/content/medical_mcq_metrics.csv",
    index=False,
)

test_analysis.to_csv(
    "/content/medical_mcq_test_predictions.csv",
    index=False,
)

topic_metrics.to_csv(
    "/content/medical_mcq_topic_metrics.csv",
    index=False,
)

ARCHIVE_PATH = shutil.make_archive(
    "/content/medical_mcq_final_model",
    "zip",
    FINAL_MODEL_DIR,
)

print("Модель:", FINAL_MODEL_DIR)
print("Архив:", ARCHIVE_PATH)

# Источники и воспроизводимость

### Датасет
`drkolesnikov/russian-nmo-medical-mcq`  
https://huggingface.co/datasets/drkolesnikov/russian-nmo-medical-mcq

### Модель
`cointegrated/rubert-tiny2`  
https://huggingface.co/cointegrated/rubert-tiny2

### Hugging Face Transformers: Multiple Choice
https://huggingface.co/docs/transformers/tasks/multiple_choice

### Возможные продолжения
- увеличить train-выборку;
- сравнить несколько русскоязычных Transformer-моделей;
- исследовать multiple-correct вопросы;
- добавить адаптивное формирование тестов по слабым темам;
- интегрировать прототип в образовательную систему.